# PathoVision - data-integrity and robustness experiments, one click

**How to use:** *Runtime > Change runtime type > L4 GPU*, then *Runtime > Run all*.

You are asked for input only at the start: Google Drive authorization and, if the raw Kaggle images are not on this machine yet, your Kaggle API token (kaggle.com > Settings > API > Create New Token; paste it into the hidden field). After that everything runs unattended (about 14 h on an L4):

| Step | What | Approx. time |
|---|---|---|
| 15-16 | label/provenance audit and CPU re-analysis | 5 min |
| 21-22 | frozen DINOv2 / ImageNet probes, border masking, Grad-CAM deletion test | 1 h |
| 19 | strict-label retraining (M4, M1 x 3 seeds) | 2 h |
| 18 | matched fine-tuning of the four backbones x 3 seeds | 3 h |
| 17 | nested 5-fold cross-validation (M1, M4) | 4 h |
| 20 | inherited, non-deduplicated split (M4, M1 x 3 seeds) | 4 h |
| 23 + bundle | tables and `results_bundle.zip` | 5 min |

Keep this browser tab open. If the session ends before finishing, open the notebook again and choose *Run all* again: finished work is detected and skipped. A failed step does not stop the others; the summary at the end lists what failed. The final bundle is saved to Google Drive (`pathovision_reanalysis/results_bundle.zip`) and downloaded automatically.

In [ ]:
#@title 1. Drive, repository, dependencies
import os, sys, subprocess, pathlib, time
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/pathovision_reanalysis'   # runs/ and results/ (persistent)
DATA = '/content/pv_data'                                 # images (fast local disk)
REPO = '/content/pathovision_model'
GITHUB_REPO = 'https://github.com/alexroehrs/pathovision_model.git'
os.makedirs(ROOT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
os.environ['REANALYSIS_ROOT'] = ROOT
os.environ['REANALYSIS_DATA_ROOT'] = DATA
os.environ['PYTHONPATH'] = REPO

def run(cmd, cwd=None, log=None):
    """Run a shell command, stream its output, return the exit code."""
    print(f'\n$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    fh = open(log, 'a') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    return p.returncode

if not os.path.exists(REPO):
    run(f'git clone --depth 1 {GITHUB_REPO} {REPO}')
else:
    run('git pull --ff-only', cwd=REPO)
os.chdir(REPO)
run('pip install -q -r requirements.txt statsmodels 2>&1 | tail -n 2')
run('pip install -q -U kaggle 2>&1 | tail -n 1')
import torch
assert torch.cuda.is_available(), 'No GPU: choose Runtime > Change runtime type > L4 GPU and run all again.'
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
#@title 2. Kaggle token (asked only if the raw images are missing on this machine)
raw = pathlib.Path(DATA) / 'raw'
def have_raw():
    return raw.exists() and all(any((raw / k).rglob('*.jp*g')) for k in ('skin_diseases_youssef', 'dermnet_goel'))
kdir = pathlib.Path.home() / '.kaggle'
need_token = (not have_raw()) and not os.environ.get('KAGGLE_API_TOKEN') \
             and not (kdir / 'access_token').exists() and not (kdir / 'kaggle.json').exists()
if need_token:
    from getpass import getpass
    token = getpass('Kaggle API token (input is hidden; press Enter to skip step 20): ').strip()
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
        kdir.mkdir(exist_ok=True)
        (kdir / 'access_token').write_text(token); os.chmod(kdir / 'access_token', 0o600)
        print('token stored for this session only')
    else:
        print('no token: step 20 (inherited split) will be skipped')
    del token
else:
    print('raw images or Kaggle credentials already available')

In [ ]:
#@title 3. Restore (or build) the datasets
archive = pathlib.Path(ROOT) / 'datasets.tar'
if archive.exists() and not (pathlib.Path(DATA) / 'data' / 'test').exists():
    print('restoring datasets from Drive ...')
    run(f'tar -xf "{archive}" -C "{DATA}"')
if not (pathlib.Path(DATA) / 'data' / '.reanalysis_done.json').exists():
    run('python reanalysis/s01_build_internal.py')
if not (pathlib.Path(DATA) / 'external_unified' / '.reanalysis_done').exists():
    run('python reanalysis/s02_build_external.py')
if not archive.exists():
    run(f'tar -cf "{archive}" -C "{DATA}" data external_unified')
assert (pathlib.Path(DATA) / 'data' / 'test').exists(), 'internal dataset missing'
print('datasets ready')

In [ ]:
#@title 4. Run every experiment (unattended, ~14 h)
LOG = str(pathlib.Path(ROOT) / 'run_all.log')
can_raw = have_raw() or bool(os.environ.get('KAGGLE_API_TOKEN')) or (kdir / 'access_token').exists() or (kdir / 'kaggle.json').exists()
STEPS = [
    ('15 label audit',              'python reanalysis/s15_label_audit.py'),
    ('16 CPU re-analysis',          'python reanalysis/s16_reanalysis.py'),
    ('21 frozen-feature probes',    'python reanalysis/s21_foundation_probe.py'),
    ('22 faithfulness',             'python reanalysis/s22_faithfulness.py'),
    ('19 strict-label retraining',  'python reanalysis/s19_strict_retrain.py'),
    ('18 matched fine-tuning',      'python reanalysis/s18_matched_finetune.py'),
    ('17 nested cross-validation',  'python reanalysis/s17_nested_cv.py'),
    ('20 inherited split',          'python reanalysis/s20_leaky_split.py' if can_raw else None),
    ('23 integrity tables',         'python reanalysis/s23_integrity_tables.py'),
]
status = {}
t_all = time.time()
for name, cmd in STEPS:
    if cmd is None:
        status[name] = 'skipped (no Kaggle token)'; continue
    t0 = time.time()
    with open(LOG, 'a') as f: f.write(f'\n===== {time.strftime("%Y-%m-%d %H:%M")} {name} =====\n')
    rc = run(cmd, log=LOG)
    status[name] = ('ok' if rc == 0 else f'FAILED (exit {rc})') + f', {(time.time() - t0) / 60:.0f} min'
    print(f'\n>>> {name}: {status[name]}\n', flush=True)
print('\n========== SUMMARY ==========')
for k, v in status.items(): print(f'{k:32s} {v}')
print(f'total {(time.time() - t_all) / 3600:.1f} h; full log: {LOG}')

In [ ]:
#@title 5. Bundle the results (tables, figures, predictions, embeddings - no checkpoints) and download
import shutil
bundle = pathlib.Path(ROOT) / 'results_bundle'
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(pathlib.Path(ROOT) / 'results', bundle / 'results')
for p in pathlib.Path(ROOT, 'runs').rglob('*'):
    if p.is_file() and (p.suffix in ('.csv', '.json', '.yaml') or p.name.startswith('emb_')):
        dst = bundle / 'runs' / p.relative_to(pathlib.Path(ROOT, 'runs')); dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, dst)
shutil.copy2(LOG, bundle / 'run_all.log')
zip_path = shutil.make_archive(str(pathlib.Path(ROOT) / 'results_bundle'), 'zip', bundle)
print('bundle written to Drive:', zip_path, f'({os.path.getsize(zip_path)/1e6:.1f} MB)')
for k, v in status.items(): print(f'{k:32s} {v}')
try:
    from google.colab import files; files.download(zip_path)
except Exception as e:
    print('automatic download not possible (', e, '); download it from Google Drive > pathovision_reanalysis')